# Person 3 — Random Forest Experiments

This notebook runs and presents the Random Forest work for the **Boosting vs Bagging** project.

It covers:

- utility and experiment tests;
- Random Forest scaling experiments;
- label-noise robustness;
- sequential versus parallel benchmark;
- coverage audit;
- generated CSV tables and figures.

> The notebook uses `n_jobs=1` for normal experiments to avoid Windows multiprocessing memory issues. The parallel benchmark uses worker values `1` and `2`.

> Running this notebook does not perform any Git commit, push, PR, or merge.

## 1. Locate the repository

Place this notebook inside the repository, preferably at:

```text
notebooks/person3_random_forest_experiments.ipynb
```

The following cell searches the current folder and its parents for a repository containing `src/` and `tests/`.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "tests").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the notebook from inside "
        "zero-bias-boosting-vs-bagging or move it into the repository."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR = REPO_ROOT / "results"

print("Repository:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print("Figures:", FIGURES_DIR)
print("Results:", RESULTS_DIR)

## 2. Command helper

Commands are executed through the same Python interpreter used by the notebook.

In [ ]:
import subprocess
from IPython.display import display, Image, Markdown
import pandas as pd

def run_command(arguments: list[str]) -> None:
    command = [sys.executable, *arguments]
    print("Running:", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )

    if completed.stdout:
        print(completed.stdout)

    if completed.stderr:
        print(completed.stderr)

    if completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: "
            + " ".join(command)
        )

## 3. Run Random Forest utility tests

In [ ]:
run_command([
    "-m",
    "pytest",
    "tests/test_rf_utils.py",
    "-q",
])

## 4. Run Random Forest experiment tests

In [ ]:
run_command([
    "-m",
    "pytest",
    "tests/test_rf_experiments.py",
    "-q",
])

## 5. Random Forest scaling experiment

The fast run evaluates small `n_estimators` and `max_depth` sweeps and saves:

- `results/rf_scaling_results.csv`
- `figures/rf_scaling_n_estimators.png`
- `figures/rf_scaling_max_depth.png`

In [ ]:
run_command([
    "-m",
    "src.experiments.rf_scaling",
    "--fast",
    "--n-jobs",
    "1",
])

In [ ]:
scaling_csv = RESULTS_DIR / "rf_scaling_results.csv"
scaling_results = pd.read_csv(scaling_csv)
display(scaling_results)

for image_name in [
    "rf_scaling_n_estimators.png",
    "rf_scaling_max_depth.png",
]:
    image_path = FIGURES_DIR / image_name
    if image_path.exists():
        display(Markdown(f"### {image_name}"))
        display(Image(filename=str(image_path)))

## 6. Label-noise robustness

The fast RF-only run avoids depending on AdaBoost integration and saves:

- `results/noise_robustness_results.csv`
- `figures/noise_robustness_breast_cancer.png`

In [ ]:
run_command([
    "-m",
    "src.experiments.noise_robustness",
    "--fast",
    "--n-jobs",
    "1",
    "--rf-only",
])

In [ ]:
noise_csv = RESULTS_DIR / "noise_robustness_results.csv"
noise_results = pd.read_csv(noise_csv)
display(noise_results)

noise_figure = FIGURES_DIR / "noise_robustness_breast_cancer.png"
if noise_figure.exists():
    display(Image(filename=str(noise_figure)))

## 7. Sequential versus parallel benchmark

This compares `n_jobs=1` and `n_jobs=2`. Avoid `n_jobs=-1` on Windows because it may create too many worker processes.

In [ ]:
run_command([
    "-m",
    "src.experiments.rf_parallel_benchmark",
    "--fast",
    "--workers",
    "1",
    "2",
])

In [ ]:
parallel_csv = RESULTS_DIR / "rf_parallel_benchmark.csv"
parallel_results = pd.read_csv(parallel_csv)
display(parallel_results)

parallel_figure = FIGURES_DIR / "rf_parallel_benchmark.png"
if parallel_figure.exists():
    display(Image(filename=str(parallel_figure)))

## 8. Coverage audit

This runs the Random Forest-related test set and writes the audit files into the root-level `results/` directory.

In [ ]:
run_command([
    "-m",
    "src.experiments.coverage_audit",
    "--rf-only",
])

In [ ]:
coverage_summary = RESULTS_DIR / "coverage_audit_summary.md"

if coverage_summary.exists():
    display(Markdown(coverage_summary.read_text(encoding="utf-8")))
else:
    print("Coverage summary was not found:", coverage_summary)

## 9. Generated files

The final cell lists the current root-level result and figure files.

In [ ]:
print("RESULT FILES")
if RESULTS_DIR.exists():
    for path in sorted(RESULTS_DIR.iterdir()):
        if path.is_file():
            print(f"- {path.relative_to(REPO_ROOT)}")

print("\nFIGURE FILES")
if FIGURES_DIR.exists():
    for path in sorted(FIGURES_DIR.iterdir()):
        if path.is_file():
            print(f"- {path.relative_to(REPO_ROOT)}")

## Notes for the final project run

- Remove `--fast` for full experiments.
- Keep ordinary Random Forest runs at `--n-jobs 1`.
- Use the parallel benchmark specifically for comparing worker counts.
- The final noise comparison can include AdaBoost after Person 2's implementation is integrated.
- The notebook only runs code and displays outputs; Git operations remain separate.